# Stochastic Volatility and Jump-Diffusion Pricing


In [1]:
import numpy as np
import pandas as pd

np.random.seed(12345)  # reproducibility for Monte Carlo


S0 = 80.0
r = 0.055
sigma_ref = 0.35 
T = 3.0 / 12.0  # 3 months in years

# Monte Carlo
N_PATHS = 50000
N_STEPS = 120 

K_atm = S0


In [3]:
# Heston model 

nu0 = 0.032      
kappa_nu = 1.85  
theta_nu = 0.045 

sigma_nu = 0.5 # standard 


def simulate_heston_paths(S0, nu0, kappa, theta, sigma_v, rho, r, T, n_paths, n_steps):
    """Heston using Euler discretization.
    """
    dt = T / n_steps

    S = np.zeros((n_paths, n_steps + 1), dtype=float)
    v = np.zeros((n_paths, n_steps + 1), dtype=float)

    S[:, 0] = S0
    v[:, 0] = nu0


    Z1 = np.random.normal(size=(n_paths, n_steps))
    Z2 = np.random.normal(size=(n_paths, n_steps))

    # Correlate Brownian motions for stock and variance
    dW1 = np.sqrt(dt) * Z1
    dW2 = np.sqrt(dt) * (rho * Z1 + np.sqrt(1.0 - rho**2) * Z2)

    for t in range(n_steps):
        v_prev = v[:, t]
        v_tilde = v_prev + kappa * (theta - v_prev) * dt + sigma_v * np.sqrt(np.maximum(v_prev, 0.0)) * dW2[:, t]
        v_next = np.maximum(v_tilde, 0.0)

        # Risk-neutral stock price 
        S_prev = S[:, t]
        S_next = S_prev * np.exp(
            (r - 0.5 * np.maximum(v_prev, 0.0)) * dt + np.sqrt(np.maximum(v_prev, 0.0)) * dW1[:, t]
        )

        v[:, t + 1] = v_next
        S[:, t + 1] = S_next

    return S, v

In [4]:
def heston_european_mc_price(S0, K, r, T, is_call, rho, n_paths=N_PATHS, n_steps=N_STEPS):
    """Monte Carlo price of a European oprtion under Heston.

    """
    S_paths, _ = simulate_heston_paths(
        S0=S0,
        nu0=nu0,
        kappa=kappa_nu,
        theta=theta_nu,
        sigma_v=sigma_nu,
        rho=rho,
        r=r,
        T=T,
        n_paths=n_paths,
        n_steps=n_steps,
    )

    S_T = S_paths[:, -1]
    if is_call:
        payoff = np.maximum(S_T - K, 0.0)
    else:
        payoff = np.maximum(K - S_T, 0.0)

    discount_factor = np.exp(-r * T)
    price = discount_factor * payoff.mean()
    std_error = discount_factor * payoff.std(ddof=1) / np.sqrt(n_paths)
    return price, std_error

## European Call Pricing under Heston Dynamics

In [7]:
rho_q5 = -0.30
rho_q6 = -0.70

results_q5_q6 = []

for label, rho in [(" (rho = -0.30)", rho_q5), (" (rho = -0.70)", rho_q6)]:
    # Call 
    call_price, call_se = heston_european_mc_price(
        S0=S0,
        K=K_atm,
        r=r,
        T=T,
        is_call=True,
        rho=rho,
        n_paths=N_PATHS,
        n_steps=N_STEPS,
    )

    # Put 
    put_price, put_se = heston_european_mc_price(
        S0=S0,
        K=K_atm,
        r=r,
        T=T,
        is_call=False,
        rho=rho,
        n_paths=N_PATHS,
        n_steps=N_STEPS,
    )

    results_q5_q6.append(
        {
            "rho": rho,
            "Call_ATM_price": round(call_price, 2),
            "Call_SE": call_se,
            "Put_ATM_price": round(put_price, 2),
            "Put_SE": put_se,
        }
    )

q5_q6_table = pd.DataFrame(results_q5_q6)
q5_q6_table

,rho,Call_ATM_price,Call_SE,Put_ATM_price,Put_SE
0,-0.3,3.40,0.020391,2.31,0.018482
1,-0.7,3.42,0.017385,2.34,0.020218


## Delta and Gamma Estimation

In [8]:
h_bump = 0.5


def heston_european_mc_price_with_rng(S0, K, r, T, is_call, rho, n_paths, n_steps, seed):
    rng = np.random.RandomState(seed)
    dt = T / n_steps

    S = np.zeros((n_paths, n_steps + 1), dtype=float)
    v = np.zeros((n_paths, n_steps + 1), dtype=float)

    S[:, 0] = S0
    v[:, 0] = nu0

    Z1 = rng.normal(size=(n_paths, n_steps))
    Z2 = rng.normal(size=(n_paths, n_steps))

    dW1 = np.sqrt(dt) * Z1
    dW2 = np.sqrt(dt) * (rho * Z1 + np.sqrt(1.0 - rho**2) * Z2)

    for t in range(n_steps):
        v_prev = v[:, t]
        v_tilde = v_prev + kappa_nu * (theta_nu - v_prev) * dt + sigma_nu * np.sqrt(np.maximum(v_prev, 0.0)) * dW2[:, t]
        v_next = np.maximum(v_tilde, 0.0)

        S_prev = S[:, t]
        S_next = S_prev * np.exp(
            (r - 0.5 * np.maximum(v_prev, 0.0)) * dt + np.sqrt(np.maximum(v_prev, 0.0)) * dW1[:, t]
        )

        v[:, t + 1] = v_next
        S[:, t + 1] = S_next

    S_T = S[:, -1]
    if is_call:
        payoff = np.maximum(S_T - K, 0.0)
    else:
        payoff = np.maximum(K - S_T, 0.0)

    discount_factor = np.exp(-r * T)
    price = discount_factor * payoff.mean()
    return price

In [11]:
def heston_delta_gamma_bump(S0, K, r, T, is_call, rho, h=h_bump, n_paths=N_PATHS, n_steps=N_STEPS, base_seed=9876):
    def price_for_S(spot):
        return heston_european_mc_price_with_rng(
            S0=spot,
            K=K,
            r=r,
            T=T,
            is_call=is_call,
            rho=rho,
            n_paths=n_paths,
            n_steps=n_steps,
            seed=base_seed,
        )

    price_down = price_for_S(S0 - h)
    price_0 = price_for_S(S0)
    price_up = price_for_S(S0 + h)

    delta = (price_up - price_down) / (2.0 * h)
    gamma = (price_up - 2.0 * price_0 + price_down) / (h ** 2)
    return price_0, delta, gamma


q7_results = []

for q_label, rho in [("", rho_q5), ("", rho_q6)]:
    for is_call, opt_type in [(True, "Call"), (False, "Put")]:
        seed = 10_000 + int(100 * abs(rho)) + (0 if is_call else 1)
        price_0, delta, gamma = heston_delta_gamma_bump(
            S0=S0,
            K=K_atm,
            r=r,
            T=T,
            is_call=is_call,
            rho=rho,
            h=h_bump,
            n_paths=N_PATHS,
            n_steps=N_STEPS,
            base_seed=seed,
        )

        q7_results.append(
            {
                "rho": rho,
                "Option_type": opt_type,
                "Price_MC": price_0,
                "Delta": delta,
                "Gamma": gamma,
            }
        )

q7_greeks_table = pd.DataFrame(q7_results)
q7_greeks_table_display = q7_greeks_table.copy()
q7_greeks_table_display["Price_MC"] = q7_greeks_table_display["Price_MC"].round(4)
q7_greeks_table_display["Delta"] = q7_greeks_table_display["Delta"].round(4)
q7_greeks_table_display["Gamma"] = q7_greeks_table_display["Gamma"].round(4)
q7_greeks_table_display

,rho,Option_type,Price_MC,Delta,Gamma
0,-0.3,Call,3.4288,0.6207,0.0564
1,-0.3,Put,2.3431,-0.3822,0.0582
2,-0.7,Call,3.4611,0.6575,0.0492
3,-0.7,Put,2.3709,-0.3436,0.0494


### – Merton jump-diffusion model

In [20]:
import numpy as np
import pandas as pd

S0, r, T, sigma_ref = 100.0, 0.05, 1.0, 0.2
N_PATHS, N_STEPS = 10000, 252
mu_J, delta_J, lambda_J = -0.5, 0.22, 0.75
B_barrier, K_q15 = 65.0, 65.0

def simulate_merton_paths(S0, r, sigma, T, n_paths, n_steps, mu_J, delta_J, lambda_J):
    dt = T / n_steps
    k = np.exp(mu_J + 0.5 * delta_J**2) - 1.0
    S = np.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0
    for t in range(n_steps):
        Z = np.random.normal(size=n_paths)
        N_t = np.random.poisson(lambda_J * dt, size=n_paths)
        jump_log = np.where(
            N_t > 0,
            np.random.normal(loc=mu_J * N_t, scale=delta_J * np.sqrt(N_t)),
            0.0
        )
        drift = (r - lambda_J * k - 0.5 * sigma**2) * dt
        S[:, t + 1] = S[:, t] * np.exp(drift + sigma * np.sqrt(dt) * Z) * np.exp(jump_log)
    return S

def merton_plain_put_price(S0, K, r, T, sigma, mu_J, delta_J, lambda_J):
    S_paths = simulate_merton_paths(S0, r, sigma, T, N_PATHS, N_STEPS, mu_J, delta_J, lambda_J)
    payoff = np.maximum(K - S_paths[:, -1], 0.0)
    price = np.exp(-r * T) * payoff.mean()
    se = np.exp(-r * T) * payoff.std(ddof=1) / np.sqrt(N_PATHS)
    return price, se

def merton_down_in_put_price(S0, K, B, r, T, sigma, mu_J, delta_J, lambda_J):
    S_paths = simulate_merton_paths(S0, r, sigma, T, N_PATHS, N_STEPS, mu_J, delta_J, lambda_J)
    hit_barrier = (S_paths <= B).any(axis=1)
    payoff = np.where(hit_barrier, np.maximum(K - S_paths[:, -1], 0.0), 0.0)
    price = np.exp(-r * T) * payoff.mean()
    se = np.exp(-r * T) * payoff.std(ddof=1) / np.sqrt(N_PATHS)
    return price, se

price_plain, se_plain = merton_plain_put_price(S0, K_q15, r, T, sigma_ref, mu_J, delta_J, lambda_J)
price_di, se_di = merton_down_in_put_price(S0, K_q15, B_barrier, r, T, sigma_ref, mu_J, delta_J, lambda_J)

results = [
    {"Question": "European Put",  "K": K_q15, "Barrier": "N/A", "Price": f"{price_plain:.4f}", "Std Error": f"{se_plain:.5f}"},
    {"Question": "Down-In Put",  "K": K_q15, "Barrier": B_barrier, "Price": f"{price_di:.4f}", "Std Error": f"{se_di:.5f}"}
]

q15_table = pd.DataFrame(results)
print(q15_table.to_string(index=False))

    Question    K Barrier  Price Std Error
European Put 65.0     N/A 3.8785   0.09655
 Down-In Put 65.0    65.0 3.8876   0.09722
